# 03 — Modelagem com Random Forest Regressor

Este notebook tem como objetivo treinar um modelo de regressão utilizando Random Forest para prever a temperatura do ar.

A base utilizada foi gerada no notebook de pré-processamento e já está dividida temporalmente em treino e teste. Dessa forma, o modelo é treinado com dados de anos anteriores e avaliado com dados de anos posteriores, evitando vazamento temporal.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline

## 1. Inicialização da SparkSession

A `SparkSession` é utilizada para carregar as bases em Parquet e executar o pipeline de modelagem com PySpark MLlib.

In [ ]:
spark = (
    SparkSession.builder
    .appName("Random_Forest_Weather_SP")
    .getOrCreate()
)

spark

## 2. Carregamento das bases de treino e teste

As bases de treino e teste foram geradas no notebook de pré-processamento.

A separação foi feita de forma temporal, preservando a lógica do problema: treinar com dados de anos anteriores e testar com anos posteriores.

In [ ]:
train_path = "/home/jovyan/work/data/processed/weather_sp_train"
test_path  = "/home/jovyan/work/data/processed/weather_sp_test"

df_treino = spark.read.parquet(train_path)
df_teste  = spark.read.parquet(test_path)

print(f"Treino: {df_treino.count():,} linhas")
print(f"Teste : {df_teste.count():,} linhas")
df_treino.show(5, truncate=False)

## 3. Verificação do schema

Antes de treinar o modelo, é importante verificar se todas as variáveis estão em formato numérico e se a variável-alvo `temperatura` está presente.

In [ ]:
df_treino.printSchema()

## 4. Definição da variável-alvo e das features

A variável-alvo do projeto é `temperatura`.

As demais colunas numéricas serão usadas como variáveis explicativas. A coluna `temperatura` é removida da lista de features para evitar que o modelo use a resposta como entrada.

In [ ]:
coluna_alvo = "temperatura"

features = [c for c in df_treino.columns if c != coluna_alvo]

print("Variável-alvo:", coluna_alvo)
print(f"Features ({len(features)}):", features)

## 5. Montagem do vetor de features

Os modelos do PySpark MLlib esperam que as variáveis explicativas estejam agrupadas em uma única coluna vetorial chamada `features`.

Para isso, é utilizado o `VectorAssembler`.

In [ ]:
assembler = VectorAssembler(
    inputCols=features,
    outputCol="features",
    handleInvalid="keep"
)

## 6. Criação do modelo Random Forest

O Random Forest é um algoritmo baseado em múltiplas árvores de decisão. Ele combina várias árvores para reduzir overfitting e melhorar a capacidade de generalização.

Como o dataset é grande e o ambiente é local, os hiperparâmetros iniciais foram mantidos moderados para evitar sobrecarga no processamento.

In [ ]:
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol=coluna_alvo,
    predictionCol="prediction",
    numTrees=20,
    maxDepth=8,
    seed=42
)

## 7. Criação do pipeline

O pipeline une a etapa de montagem das features com o treinamento do modelo. Isso organiza o fluxo e facilita a reprodução do processo.

In [ ]:
pipeline = Pipeline(stages=[assembler, rf])

## 8. Treinamento do modelo

Nesta etapa, o modelo Random Forest é treinado utilizando a base de treino.

In [ ]:
modelo_rf = pipeline.fit(df_treino)

print("Treinamento do Random Forest finalizado.")

## 9. Geração das previsões

Após o treinamento, o modelo é aplicado sobre a base de teste para gerar as previsões de temperatura.

In [ ]:
predicoes_rf = modelo_rf.transform(df_teste)

predicoes_rf.select(
    "temperatura",
    "prediction"
).show(20, truncate=False)

## 10. Avaliação do modelo

O desempenho do modelo será avaliado com três métricas de regressão:

- **MAE**: erro médio absoluto.
- **RMSE**: raiz do erro quadrático médio.
- **R²**: coeficiente de determinação.

Quanto menores MAE e RMSE, melhor. Quanto mais próximo de 1 o R², melhor.

In [ ]:
avaliador_mae = RegressionEvaluator(
    labelCol=coluna_alvo,
    predictionCol="prediction",
    metricName="mae"
)

avaliador_rmse = RegressionEvaluator(
    labelCol=coluna_alvo,
    predictionCol="prediction",
    metricName="rmse"
)

avaliador_r2 = RegressionEvaluator(
    labelCol=coluna_alvo,
    predictionCol="prediction",
    metricName="r2"
)

mae = avaliador_mae.evaluate(predicoes_rf)
rmse = avaliador_rmse.evaluate(predicoes_rf)
r2 = avaliador_r2.evaluate(predicoes_rf)

print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²: {r2:.4f}")

## 11. Comparação entre valores reais e previstos

Nesta etapa são exibidos alguns exemplos de temperatura real e temperatura prevista pelo modelo.

In [ ]:
predicoes_rf.select(
    "ano",
    "mes",
    "dia",
    "hora_num",
    "temperatura",
    F.round("prediction", 2).alias("temperatura_prevista")
).show(30, truncate=False)

## 12. Análise do erro de previsão

É criada uma coluna com o erro absoluto entre a temperatura real e a temperatura prevista.

Essa análise ajuda a entender o tamanho médio dos desvios cometidos pelo modelo.

In [ ]:
predicoes_rf = predicoes_rf.withColumn(
    "erro_absoluto",
    F.abs(F.col("temperatura") - F.col("prediction"))
)

predicoes_rf.select(
    "temperatura",
    F.round("prediction", 2).alias("temperatura_prevista"),
    F.round("erro_absoluto", 2).alias("erro_absoluto")
).show(30, truncate=False)

## 13. Estatísticas do erro

São calculadas estatísticas gerais do erro absoluto para avaliar o comportamento do modelo.

In [ ]:
predicoes_rf.select(
    F.round(F.avg("erro_absoluto"), 4).alias("media_erro_absoluto"),
    F.round(F.min("erro_absoluto"), 4).alias("menor_erro"),
    F.round(F.max("erro_absoluto"), 4).alias("maior_erro"),
    F.round(F.stddev("erro_absoluto"), 4).alias("desvio_padrao_erro")
).show(truncate=False)

## 14. Importância das variáveis

O Random Forest permite verificar a importância relativa de cada variável utilizada no treinamento.

Essa análise ajuda a entender quais atributos mais influenciaram a previsão da temperatura.

In [ ]:
modelo_rf_treinado = modelo_rf.stages[-1]

importancias = modelo_rf_treinado.featureImportances

linhas_importancia = []

for feature, importancia in zip(features, importancias):
    linhas_importancia.append((feature, float(importancia)))

df_importancias = spark.createDataFrame(
    linhas_importancia,
    ["feature", "importancia"]
)

df_importancias.orderBy(F.desc("importancia")).show(50, truncate=False)

## 15. Avaliação por ano

Como a separação foi feita de forma temporal, é útil verificar o erro médio por ano na base de teste.

In [ ]:
predicoes_rf.groupBy("ano").agg(
    F.count("*").alias("total_registros"),
    F.round(F.avg("temperatura"), 2).alias("temperatura_media_real"),
    F.round(F.avg("prediction"), 2).alias("temperatura_media_prevista"),
    F.round(F.avg("erro_absoluto"), 4).alias("mae_por_ano")
).orderBy("ano").show(truncate=False)

## 16. Avaliação por mês

A avaliação por mês permite observar se o modelo apresenta maior erro em determinados períodos do ano.

In [ ]:
predicoes_rf.groupBy("mes").agg(
    F.count("*").alias("total_registros"),
    F.round(F.avg("temperatura"), 2).alias("temperatura_media_real"),
    F.round(F.avg("prediction"), 2).alias("temperatura_media_prevista"),
    F.round(F.avg("erro_absoluto"), 4).alias("mae_por_mes")
).orderBy("mes").show(12, truncate=False)

## 17. Salvamento das predições

As previsões do modelo são salvas em formato Parquet para posterior comparação com os outros modelos.

In [ ]:
predicoes_path = "/home/jovyan/work/data/processed/predicoes_random_forest"

predicoes_rf.select(
    "ano",
    "mes",
    "dia",
    "hora_num",
    "temperatura",
    "prediction",
    "erro_absoluto"
).write.mode("overwrite").parquet(predicoes_path)

print(f"Predições do Random Forest salvas em: {predicoes_path}")

## 18. Salvamento das métricas

As métricas do modelo são organizadas em um DataFrame Spark e salvas para comparação posterior com os demais modelos.

In [ ]:
metricas_rf = [
    ("Random Forest", float(mae), float(rmse), float(r2))
]

df_metricas_rf = spark.createDataFrame(
    metricas_rf,
    ["modelo", "mae", "rmse", "r2"]
)

df_metricas_rf.show(truncate=False)

metricas_path = "/home/jovyan/work/data/processed/metricas_random_forest"

df_metricas_rf.write.mode("overwrite").parquet(metricas_path)

print(f"Métricas do Random Forest salvas em: {metricas_path}")

## 19. Salvamento do modelo

O modelo treinado é salvo para que possa ser reutilizado posteriormente sem necessidade de novo treinamento.

In [ ]:
modelo_path = "/home/jovyan/work/models/random_forest_weather"

modelo_rf.write().overwrite().save(modelo_path)

print(f"Modelo Random Forest salvo em: {modelo_path}")

## Conclusão

Neste notebook foi treinado um modelo Random Forest Regressor para prever a temperatura do ar.

Foram realizadas as seguintes etapas:

- Carregamento das bases de treino e teste.
- Definição das features e da variável-alvo.
- Criação de pipeline com `VectorAssembler` e `RandomForestRegressor`.
- Treinamento do modelo.
- Geração de previsões.
- Avaliação com MAE, RMSE e R².
- Análise de erro por ano e por mês.
- Verificação da importância das variáveis.
- Salvamento das predições, métricas e modelo treinado.

Os resultados deste modelo serão posteriormente comparados com a Rede Neural e com o terceiro modelo escolhido.